In [ ]:
# Cell 1: Ingress, Filter, Export

# --- 1. SETUP & AUTHENTICATION ---
# Install required libraries for Google Sheets integration
# !pip install gspread google-auth-oauthlib pandas -q

import pandas as pd
import gspread
from google.colab import auth
from google.colab import drive
import os

print("Step 1: Initiating Authentication and Drive Mount...")

# Authenticate user for Google API access
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# Mount Google Drive to access the file system
drive.mount('/content/drive')

print("Authentication successful and Google Drive mounted.")

# --- 2. CONFIGURATION & DATA INGRESS ---
# Define canonical source and destination parameters
GSHEET_NAME = 'raw_Offers'
WORKSHEET_NAME = 'diamond_offers'
# Note: '/content/drive/My Drive/' is the Colab path for your "My Drive" folder.
OUTPUT_DIRECTORY = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILE_VALID = 'filtered_valid_offers.csv'
OUTPUT_FILE_OUTLIERS = 'filtered_outlier_offers.csv'

# Define the geospatial bounding box for Mexico City
MIN_LAT, MAX_LAT = 19.0, 19.8
MIN_LON, MAX_LON = -99.5, -98.7

print(f"\nStep 2: Accessing GSheet '{GSHEET_NAME}' -> Worksheet '{WORKSHEET_NAME}'...")

try:
    # Open the spreadsheet and select the worksheet
    worksheet = gc.open(GSHEET_NAME).worksheet(WORKSHEET_NAME)

    # Fetch all data and load into a pandas DataFrame
    data = worksheet.get_all_records()
    df = pd.DataFrame(data)

    print(f"Successfully ingested {len(df)} records.")
    # Ensure coordinate columns are numeric, coercing errors to NaN
    coord_cols = ['pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon']
    for col in coord_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Report rows with non-numeric coordinates, as they will be treated as outliers
    invalid_coords = df[coord_cols].isnull().any(axis=1).sum()
    if invalid_coords > 0:
        print(f"Warning: Found {invalid_coords} rows with missing or non-numeric coordinate data.")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"ERROR: Spreadsheet '{GSHEET_NAME}' not found. Check name and permissions.")
except gspread.exceptions.WorksheetNotFound:
    print(f"ERROR: Worksheet '{WORKSHEET_NAME}' not found in the spreadsheet.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


# --- 3. GEOSPATIAL FILTRATION ---
if 'df' in locals():
    print("\nStep 3: Executing Geospatial Bounding Box Filtration...")

    # Handle potential missing coordinates by filling with a value that is out of bounds (e.g., 0)
    df_filtered = df.fillna({col: 0 for col in coord_cols})

    # Create boolean masks to identify records where coordinates are within the valid bounds.
    is_pickup_ok = df_filtered['pickup_lat'].between(MIN_LAT, MAX_LAT) & df_filtered['pickup_lon'].between(MIN_LON, MAX_LON)
    is_dropoff_ok = df_filtered['dropoff_lat'].between(MIN_LAT, MAX_LAT) & df_filtered['dropoff_lon'].between(MIN_LON, MAX_LON)

    # A record is valid ONLY if BOTH pickup AND dropoff are within the box.
    valid_mask = is_pickup_ok & is_dropoff_ok

    # Segregate the original dataframe (with potential NaNs) using the mask
    df_valid = df[valid_mask].copy()
    df_outliers = df[~valid_mask].copy()

    print("\n--- FILTRATION SUMMARY ---")
    print(f"Identified {len(df_valid)} records within CDMX operational bounds.")
    print(f"Isolated {len(df_outliers)} outlier records for forensic investigation.")


    # --- 4. EXPORT RESULTS ---
    print(f"\nStep 4: Exporting results to '{OUTPUT_DIRECTORY}'...")

    # Ensure the output directory exists
    os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

    # Define full output paths
    path_valid = os.path.join(OUTPUT_DIRECTORY, OUTPUT_FILE_VALID)
    path_outliers = os.path.join(OUTPUT_DIRECTORY, OUTPUT_FILE_OUTLIERS)

    # Export the dataframes to CSV files
    df_valid.to_csv(path_valid, index=False)
    df_outliers.to_csv(path_outliers, index=False)

    print(f"  -> Valid records saved to: {path_valid}")
    print(f"  -> Outlier records saved to: {path_outliers}")
    print("\nPhase 1 Complete.")

Step 1: Initiating Authentication and Drive Mount...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Authentication successful and Google Drive mounted.

Step 2: Accessing GSheet 'raw_Offers' -> Worksheet 'diamond_offers'...
Successfully ingested 4766 records.

Step 3: Executing Geospatial Bounding Box Filtration...

--- FILTRATION SUMMARY ---
Identified 4141 records within CDMX operational bounds.
Isolated 625 outlier records for forensic investigation.

Step 4: Exporting results to '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'...
  -> Valid records saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/filtered_valid_offers.csv
  -> Outlier records saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/filtered_outlier_offers.csv

Phase 1 Complete.


In [ ]:
# ==============================================================================
# CELL: Audit Rows with Corrupted Coordinate Data
# ==============================================================================
# Objective: Isolate and inspect the records that failed the numeric conversion,
# indicating corrupted or missing coordinate entries.

print("--- AUDIT: Corrupted Coordinate Records ---")

# This audit logic must run AFTER the main DataFrame 'df' is created and
# the coordinate columns have been converted to numeric types.

# Define the columns that were subjected to numeric conversion
coord_cols_to_audit = ['pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon']

# The 'errors=coerce' argument in the previous cell's pd.to_numeric function
# turns any non-numeric value (e.g., empty strings, text) into NaN (Not a Number).
# We can now create a mask to find any row where at least one of these columns is NaN.
audit_mask = df[coord_cols_to_audit].isnull().any(axis=1)

# Filter the original DataFrame to isolate only the anomalous records.
corrupted_records_df = df[audit_mask]

if not corrupted_records_df.empty:
    print(f"Found {len(corrupted_records_df)} records with missing or non-numeric coordinate data.")
    print("Displaying records for inspection:")

    # To ensure all relevant data is visible, we will select the ID, address, and coordinate columns.
    display_cols = [
        'offer_id',
        'pickup_address',
        'pickup_lat',
        'pickup_lon',
        'dropoff_address',
        'dropoff_lat',
        'dropoff_lon'
    ]

    # Filter for columns that actually exist in the DataFrame to prevent errors
    existing_display_cols = [col for col in display_cols if col in corrupted_records_df.columns]

    display(corrupted_records_df[existing_display_cols])

else:
    print("Audit complete. No records with corrupted coordinates were found in the dataset.")

--- AUDIT: Corrupted Coordinate Records ---
Found 2 records with missing or non-numeric coordinate data.
Displaying records for inspection:


,offer_id,pickup_address,pickup_lat,pickup_lon,dropoff_address,dropoff_lat,dropoff_lon
2,OF00003,,NaN,NaN,,NaN,NaN
1493,OF01494,,NaN,NaN,"Carlos Graef Fernández, Col Lomas de Santa Fe ...",19.359479,-99.270727


In [ ]:
# ==============================================================================
# CELL: Forensic Reverse Geocoding of Outliers (Corrected for KeyError)
# ==============================================================================
# Objective: Preserve original columns, perform reverse geocoding on each
# coordinate pair, and append the results as two new columns.

# --- 0. INSTALL DEPENDENCIES ---
!pip install googlemaps -q

import pandas as pd
import googlemaps
import os
import time
from google.colab import userdata
from tqdm.auto import tqdm

print("--- Forensic Investigation Initializing ---")

# --- 1. SETUP & CONFIGURATION ---
print("Step 1: Initializing Google Maps Client and file paths...")
try:
    API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
    gmaps = googlemaps.Client(key=API_KEY)
    print(" -> Google Maps client initialized successfully.")
except userdata.SecretNotFoundError:
    raise ValueError("CRITICAL ERROR: API key 'GOOGLE_MAPS_API_KEY' not found in Colab Secrets. Mission aborted.")

ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
INPUT_FILE = os.path.join(ANALYSIS_PATH, 'filtered_outlier_offers.csv')
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'forensic_results.csv')

# --- 2. LOAD DATA & PREPARE FOR RESUMPTION ---
print("\nStep 2: Loading outlier data and checking for previous progress...")
outliers_df = pd.read_csv(INPUT_FILE)
outliers_df['offer_id'] = outliers_df['offer_id'].astype(str)
processed_ids = set()
results_data = []

if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Preparing to resume...")
    results_df = pd.read_csv(OUTPUT_FILE)
    results_df['offer_id'] = results_df['offer_id'].astype(str)
    processed_ids = set(results_df['offer_id'])
    results_data = results_df.to_dict('records')
    print(f" -> {len(processed_ids)} records have already been processed and will be skipped.")

records_to_process = outliers_df[~outliers_df['offer_id'].isin(processed_ids)].copy()

# --- 3. REVERSE GEOCODING ENGINE ---
print(f"\nStep 3: Commencing reverse geocoding for {len(records_to_process)} remaining records...")

if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))

    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Processing offer_id: {row['offer_id']}")

        # CORRECTION: Start with the original row data to preserve all columns.
        result_row = row.to_dict()

        # --- Process Pickup Coordinates ---
        try:
            if pd.notna(row['pickup_lat']) and pd.notna(row['pickup_lon']):
                reverse_geocode_result = gmaps.reverse_geocode((row['pickup_lat'], row['pickup_lon']))
                # Add the new verified location column.
                result_row['verified_pickup_location'] = reverse_geocode_result[0]['formatted_address'] if reverse_geocode_result else "No Address Found"
            else:
                result_row['verified_pickup_location'] = "Invalid/Missing Coordinates"
        except Exception as e:
            result_row['verified_pickup_location'] = f"API Error: {e}"

        # --- Process Dropoff Coordinates ---
        try:
            if pd.notna(row['dropoff_lat']) and pd.notna(row['dropoff_lon']):
                reverse_geocode_result = gmaps.reverse_geocode((row['dropoff_lat'], row['dropoff_lon']))
                # Add the new verified location column.
                result_row['verified_dropoff_location'] = reverse_geocode_result[0]['formatted_address'] if reverse_geocode_result else "No Address Found"
            else:
                result_row['verified_dropoff_location'] = "Invalid/Missing Coordinates"
        except Exception as e:
            result_row['verified_dropoff_location'] = f"API Error: {e}"

        results_data.append(result_row)

        # --- Checkpoint Trigger ---
        if (i + 1) % 25 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Progress saved to checkpoint.")

    print("\n--- Forensic Geocoding Complete ---")
else:
    print("\nAll records have already been processed.")

# --- 4. FINAL OUTPUT ---
final_results_df = pd.DataFrame(results_data)
final_results_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nInvestigation complete. Full results are saved to: {OUTPUT_FILE}")
print("Displaying a sample of the results:")
display(final_results_df.head())

--- Forensic Investigation Initializing ---
Step 1: Initializing Google Maps Client and file paths...
 -> Google Maps client initialized successfully.

Step 2: Loading outlier data and checking for previous progress...

Step 3: Commencing reverse geocoding for 625 remaining records...


  0%|          | 0/625 [00:00<?, ?it/s]


--- Forensic Geocoding Complete ---

Investigation complete. Full results are saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/forensic_results.csv
Displaying a sample of the results:


,offer_id,session_fk,ocr_fk,image_content_hash,offer_timestamp,product_category_fk,upfront_fare,time_to_pickup_sec,dist_to_pickup_km,est_trip_time_sec,...,inferred_agent_bearing,inferred_agent_speed_mps,interpolation_quality_fk,is_imputed,special_note_raw,comment_1,comment_2,record_status_fk,verified_pickup_location,verified_dropoff_location
0,OF00003,SID0001,0343d4234ef85474adcff2430891b7473c274bbc9cb30f...,64f34e64aaed7113e732d1434d857ac4de639b89626cf4...,2025:08:22 06:45:28,uberx,136.53,NaN,NaN,NaN,...,NaN,NaN,NaN,True,NaN,NaN,NaN,invalid_non_offer,Invalid/Missing Coordinates,Invalid/Missing Coordinates
1,OF00014,SID0001,d564072cb9ddc73fa66f041f3dff9a3ff335e6015811e2...,202ebcba31d59b2d5491787367fc8b44b9319f9e3ed639...,2025:08:22 08:10:05,uberx,167.98,480.0,2.6,3120.0,...,212.679706,4.505874,INTERPOLATED,False,Identidad verificada (Identity verified),NaN,NaN,valid,"Claveles 4383, S3004 Santa Fe de la Vera Cruz,...","Canal de Tezontle 1512, Dr Alfonso Ortiz Tirad..."
2,OF00017,SID0001,4fbb61fec9e45a38e897144b09634b1bfa39c28434688b...,0dd678e8a9e09167416354dbf70215772718b4db7f252a...,2025:08:22 08:12:06,uberx,102.37,540.0,4.0,1200.0,...,212.679706,0.000000,EXTRAPOLATED_START,False,"Identidad verificada (Identity verified), +$10...",NaN,NaN,valid,"110 W Marcy St, Santa Fe, NM 87501, USA",Periferico Sur 9637 Torre Angeles Consultorio ...
3,OF00019,SID0001,5175efdec3c45978d14d0fb83f3caa274834554ee81890...,66d0198cc3377f1f6819351858fd6c4383d7bb3386e169...,2025:08:22 08:12:25,uberx,36.58,540.0,3.5,480.0,...,212.679706,0.000000,EXTRAPOLATED_START,False,"Identidad verificada (Identity verified), +$10...",NaN,NaN,valid,"110 W Marcy St, Santa Fe, NM 87501, USA","Prol. P.º de la Reforma 400, Santa Fe, Paseo d..."
4,OF00020,SID0001,ae2e09048576d623b78950ee7f0ea28478f1f1305fca07...,a200fbeb4bc520e2fc0b1dc20832ea9bfbe4d1734ba190...,2025:08:22 08:12:39,uberx,116.01,1140.0,9.2,1380.0,...,212.679706,0.000000,EXTRAPOLATED_START,False,Identidad verificada (Identity verified),NaN,NaN,valid,"C. Comandante Ruiz Marcet, 17, 11100 San Ferna...","P.º de los Tamarindos 60-PB, Bosques de las Lo..."


In [ ]:
# ==============================================================================
# CELL: Geocode Remediation Engine
# ==============================================================================
# Objective: For each outlier, re-attempt forward geocoding on the original
# address strings with a strict bounding box for Mexico City.

# --- 0. INSTALL DEPENDENCIES ---
!pip install googlemaps -q

import pandas as pd
import googlemaps
import os
from google.colab import userdata
from tqdm.auto import tqdm

print("--- Geocode Remediation Engine Initializing ---")

# --- 1. SETUP & CONFIGURATION ---
print("Step 1: Initializing Google Maps Client and file paths...")
try:
    API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
    gmaps = googlemaps.Client(key=API_KEY)
    print(" -> Google Maps client initialized successfully.")
except userdata.SecretNotFoundError:
    raise ValueError("CRITICAL ERROR: API key 'GOOGLE_MAPS_API_KEY' not found in Colab Secrets.")

# Define the CDMX bounding box for the API request
cdmx_bounds = {
    'southwest': (19.0, -99.5),
    'northeast': (19.8, -98.7)
}

# Define file paths
ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
INPUT_FILE = os.path.join(ANALYSIS_PATH, 'filtered_outlier_offers.csv')
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'remediated_offers.csv')

# --- 2. LOAD DATA & PREPARE FOR RESUMPTION ---
print("\nStep 2: Loading outlier data...")
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"CRITICAL ERROR: Outlier file not found at '{INPUT_FILE}'")

outliers_df = pd.read_csv(INPUT_FILE)
# Checkpointing logic for robustness on large datasets
processed_ids = set()
results_data = []

if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    results_data = results_df.to_dict('records')
    print(f" -> {len(processed_ids)} records previously processed.")

records_to_process = outliers_df[~outliers_df['offer_id'].astype(str).isin(processed_ids)].copy()

# --- 3. FORWARD GEOCODING REMEDIATION ENGINE ---
print(f"\nStep 3: Commencing bounded forward geocoding for {len(records_to_process)} remaining records...")

if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))

    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Remediating offer_id: {row['offer_id']}")

        result_row = row.to_dict()
        # Initialize new columns for the remediated data
        result_row.update({
            'regeocoded_pickup_lat': None, 'regeocoded_pickup_lon': None,
            'regeocoded_dropoff_lat': None, 'regeocoded_dropoff_lon': None,
            'remediation_status': []
        })

        # --- Remediate Pickup Address ---
        if pd.notna(row['pickup_address']):
            try:
                geocode_result = gmaps.geocode(row['pickup_address'], bounds=cdmx_bounds)
                if geocode_result:
                    loc = geocode_result[0]['geometry']['location']
                    result_row['regeocoded_pickup_lat'] = loc['lat']
                    result_row['regeocoded_pickup_lon'] = loc['lng']
                    result_row['remediation_status'].append('PICKUP_SUCCESS')
                else:
                    result_row['remediation_status'].append('PICKUP_FAILED')
            except Exception as e:
                result_row['remediation_status'].append(f'PICKUP_API_ERROR: {e}')
        else:
            result_row['remediation_status'].append('PICKUP_NO_ADDRESS')

        # --- Remediate Dropoff Address ---
        if pd.notna(row['dropoff_address']):
            try:
                geocode_result = gmaps.geocode(row['dropoff_address'], bounds=cdmx_bounds)
                if geocode_result:
                    loc = geocode_result[0]['geometry']['location']
                    result_row['regeocoded_dropoff_lat'] = loc['lat']
                    result_row['regeocoded_dropoff_lon'] = loc['lng']
                    result_row['remediation_status'].append('DROPOFF_SUCCESS')
                else:
                    result_row['remediation_status'].append('DROPOFF_FAILED')
            except Exception as e:
                result_row['remediation_status'].append(f'DROPOFF_API_ERROR: {e}')
        else:
            result_row['remediation_status'].append('DROPOFF_NO_ADDRESS')

        result_row['remediation_status'] = ', '.join(result_row['remediation_status'])
        results_data.append(result_row)

        # --- Checkpoint Trigger ---
        if (i + 1) % 25 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Checkpoint saved.")

    print("\n--- Remediation Geocoding Complete ---")
else:
    print("\nAll records have already been processed.")

# --- 4. FINAL OUTPUT ---
final_results_df = pd.DataFrame(results_data)
final_results_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nRemediation complete. Enriched results saved to: {OUTPUT_FILE}")
print("Displaying a sample of the results for comparison:")

# Display key columns to compare original vs. remediated
display_cols = [
    'offer_id', 'pickup_address', 'pickup_lat', 'pickup_lon', 'regeocoded_pickup_lat', 'regeocoded_pickup_lon',
    'dropoff_address', 'dropoff_lat', 'dropoff_lon', 'regeocoded_dropoff_lat', 'regeocoded_dropoff_lon',
    'remediation_status'
]
display(final_results_df[[col for col in display_cols if col in final_results_df.columns]].head())

--- Geocode Remediation Engine Initializing ---
Step 1: Initializing Google Maps Client and file paths...
 -> Google Maps client initialized successfully.

Step 2: Loading outlier data...

Step 3: Commencing bounded forward geocoding for 625 remaining records...


  0%|          | 0/625 [00:00<?, ?it/s]


--- Remediation Geocoding Complete ---

Remediation complete. Enriched results saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/remediated_offers.csv
Displaying a sample of the results for comparison:


,offer_id,pickup_address,pickup_lat,pickup_lon,regeocoded_pickup_lat,regeocoded_pickup_lon,dropoff_address,dropoff_lat,dropoff_lon,regeocoded_dropoff_lat,regeocoded_dropoff_lon,remediation_status
0,OF00003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"PICKUP_NO_ADDRESS, DROPOFF_NO_ADDRESS"
1,OF00014,"Avenida Constituyentes, Camino Antiguo a Santa Fe",-31.610658,-60.697294,19.402747,-99.207907,"Canal de Tezontle 1512, Col Alfonso Ortiz Tira...",19.383654,-99.083223,19.384339,-99.076739,"PICKUP_SUCCESS, DROPOFF_SUCCESS"
2,OF00017,"Sta. Fe, Santa Fe",35.689431,-105.938194,19.386393,-99.225562,"Periferico Sur, Anillo Perif. 4121, Equipamien...",19.312828,-99.220405,19.309183,-99.217973,"PICKUP_SUCCESS, DROPOFF_SUCCESS"
3,OF00019,"Av Guillermo González Camarena, Santa Fe",35.689431,-105.938194,19.366602,-99.263853,"Prol. P.º de la Reforma 379, Santa Fe, Zedec S...",19.369405,-99.266431,19.369405,-99.266431,"PICKUP_SUCCESS, DROPOFF_SUCCESS"
4,OF00020,"Calle Granada, San Fernando Huixquilucan",36.471865,-6.196595,19.386054,-99.282716,"Paseo de los Tamarindos 60, Bosques de Las Lom...",19.388529,-99.251772,19.388529,-99.251772,"PICKUP_SUCCESS, DROPOFF_SUCCESS"


In [ ]:
# ==============================================================================
# CELL: Geocode Remediation & Verification Engine
# ==============================================================================
# Objective: For each outlier, re-geocode the address with a regional bound, then
# immediately reverse-geocode the new coordinate to verify the location.

# --- 0. INSTALL DEPENDENCIES ---
!pip install googlemaps -q

import pandas as pd
import googlemaps
import os
from google.colab import userdata
from tqdm.auto import tqdm
import time

print("--- Geocode Remediation & Verification Engine Initializing ---")

# --- 1. SETUP & CONFIGURATION ---
print("Step 1: Initializing Google Maps Client and file paths...")
try:
    API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
    gmaps = googlemaps.Client(key=API_KEY)
    print(" -> Google Maps client initialized successfully.")
except userdata.SecretNotFoundError:
    raise ValueError("CRITICAL ERROR: API key 'GOOGLE_MAPS_API_KEY' not found in Colab Secrets.")

cdmx_bounds = {'southwest': (19.0, -99.5), 'northeast': (19.8, -98.7)}
ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
INPUT_FILE = os.path.join(ANALYSIS_PATH, 'filtered_outlier_offers.csv')
# New output file name to reflect the added verification step
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'remediated_offers_verified.csv')

# --- 2. LOAD DATA & PREPARE FOR RESUMPTION ---
print("\nStep 2: Loading outlier data and checking for previous progress...")
outliers_df = pd.read_csv(INPUT_FILE)
outliers_df['offer_id'] = outliers_df['offer_id'].astype(str)
processed_ids = set()
results_data = []

if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    results_data = results_df.to_dict('records')
    print(f" -> {len(processed_ids)} records previously processed.")

records_to_process = outliers_df[~outliers_df['offer_id'].astype(str).isin(processed_ids)].copy()

# --- 3. REMEDIATION & VERIFICATION ENGINE ---
print(f"\nStep 3: Commencing remediation & verification for {len(records_to_process)} remaining records...")

if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))

    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Processing offer_id: {row['offer_id']}")

        result_row = row.to_dict()
        result_row.update({
            'regeocoded_pickup_lat': None, 'regeocoded_pickup_lon': None, 'verified_regeocoded_pickup_address': None,
            'regeocoded_dropoff_lat': None, 'regeocoded_dropoff_lon': None, 'verified_regeocoded_dropoff_address': None,
            'remediation_status': []
        })

        # --- Process Pickup Address ---
        if pd.notna(row['pickup_address']):
            try:
                # Stage 1: Forward geocode with bounds
                geocode_result = gmaps.geocode(row['pickup_address'], bounds=cdmx_bounds)
                if geocode_result:
                    loc = geocode_result[0]['geometry']['location']
                    lat, lon = loc['lat'], loc['lng']
                    result_row['regeocoded_pickup_lat'] = lat
                    result_row['regeocoded_pickup_lon'] = lon
                    result_row['remediation_status'].append('PICKUP_REMEDIATED')

                    # Stage 2: Immediately reverse geocode the new coordinate for verification
                    time.sleep(0.05) # Polite delay
                    reverse_result = gmaps.reverse_geocode((lat, lon))
                    if reverse_result:
                        result_row['verified_regeocoded_pickup_address'] = reverse_result[0]['formatted_address']
                        result_row['remediation_status'].append('PICKUP_VERIFIED')
                else:
                    result_row['remediation_status'].append('PICKUP_GEOCODE_FAILED')
            except Exception as e:
                result_row['remediation_status'].append(f'PICKUP_API_ERROR')
        else:
            result_row['remediation_status'].append('PICKUP_NO_ADDRESS')

        # --- Process Dropoff Address ---
        if pd.notna(row['dropoff_address']):
            try:
                geocode_result = gmaps.geocode(row['dropoff_address'], bounds=cdmx_bounds)
                if geocode_result:
                    loc = geocode_result[0]['geometry']['location']
                    lat, lon = loc['lat'], loc['lng']
                    result_row['regeocoded_dropoff_lat'] = lat
                    result_row['regeocoded_dropoff_lon'] = lon
                    result_row['remediation_status'].append('DROPOFF_REMEDIATED')

                    time.sleep(0.05)
                    reverse_result = gmaps.reverse_geocode((lat, lon))
                    if reverse_result:
                        result_row['verified_regeocoded_dropoff_address'] = reverse_result[0]['formatted_address']
                        result_row['remediation_status'].append('DROPOFF_VERIFIED')
                else:
                    result_row['remediation_status'].append('DROPOFF_GEOCODE_FAILED')
            except Exception as e:
                result_row['remediation_status'].append(f'DROPOFF_API_ERROR')
        else:
             result_row['remediation_status'].append('DROPOFF_NO_ADDRESS')

        result_row['remediation_status'] = ', '.join(result_row['remediation_status'])
        results_data.append(result_row)

        if (i + 1) % 10 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Checkpoint saved.")

    print("\n--- Remediation & Verification Complete ---")
else:
    print("\nAll records have already been processed.")

# --- 4. FINAL OUTPUT ---
final_results_df = pd.DataFrame(results_data)
final_results_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nFull results saved to: {OUTPUT_FILE}")
print("Displaying a sample of the verification results:")

display_cols = [
    'offer_id', 'pickup_address', 'regeocoded_pickup_lat', 'regeocoded_pickup_lon', 'verified_regeocoded_pickup_address',
    'remediation_status'
]
display(final_results_df[[col for col in display_cols if col in final_results_df.columns]].head())

--- Geocode Remediation & Verification Engine Initializing ---
Step 1: Initializing Google Maps Client and file paths...
 -> Google Maps client initialized successfully.

Step 2: Loading outlier data and checking for previous progress...

Step 3: Commencing remediation & verification for 625 remaining records...


  0%|          | 0/625 [00:00<?, ?it/s]


--- Remediation & Verification Complete ---

Full results saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/remediated_offers_verified.csv
Displaying a sample of the verification results:


,offer_id,pickup_address,regeocoded_pickup_lat,regeocoded_pickup_lon,verified_regeocoded_pickup_address,remediation_status
0,OF00003,NaN,NaN,NaN,None,"PICKUP_NO_ADDRESS, DROPOFF_NO_ADDRESS"
1,OF00014,"Avenida Constituyentes, Camino Antiguo a Santa Fe",19.402747,-99.207907,"Av Constituyentes 551BIS, 16 de Septiembre, Mi...","PICKUP_REMEDIATED, PICKUP_VERIFIED, DROPOFF_RE..."
2,OF00017,"Sta. Fe, Santa Fe",19.386393,-99.225562,"Vasco de Quiroga 1225, Sta Fé, Álvaro Obregón,...","PICKUP_REMEDIATED, PICKUP_VERIFIED, DROPOFF_RE..."
3,OF00019,"Av Guillermo González Camarena, Santa Fe",19.366602,-99.263853,"C. Guillermo Gonzalez Camarena 1205, Santa Fe,...","PICKUP_REMEDIATED, PICKUP_VERIFIED, DROPOFF_RE..."
4,OF00020,"Calle Granada, San Fernando Huixquilucan",19.386054,-99.282716,"José Francisco 1, San Fernando, 52765 Naucalpa...","PICKUP_REMEDIATED, PICKUP_VERIFIED, DROPOFF_RE..."


In [ ]:
# ==============================================================================
# CELL: Remediation & Verification Engine v2.0 (with Geospatial Jittering)
# ==============================================================================
# Objective: Re-geocode outliers, applying random jittering within the bounding
# box of ambiguous area-level results to ensure analytical integrity.

# --- 0. INSTALL DEPENDENCIES ---
!pip install googlemaps -q

import pandas as pd
import googlemaps
import os
import numpy as np
from google.colab import userdata
from tqdm.auto import tqdm
import time

print("--- Remediation & Verification Engine v2.0 Initializing ---")

# --- 1. SETUP & CONFIGURATION ---
print("Step 1: Initializing Google Maps Client and file paths...")
API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=API_KEY)
cdmx_bounds = {'southwest': (19.0, -99.5), 'northeast': (19.8, -98.7)}
ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
INPUT_FILE = os.path.join(ANALYSIS_PATH, 'filtered_outlier_offers.csv')
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'remediated_offers_verified_jittered.csv')

# --- 2. LOAD DATA & PREPARE FOR RESUMPTION ---
print("\nStep 2: Loading outlier data...")
outliers_df = pd.read_csv(INPUT_FILE)
outliers_df['offer_id'] = outliers_df['offer_id'].astype(str)
processed_ids, results_data = set(), []
if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    results_data = results_df.to_dict('records')
records_to_process = outliers_df[~outliers_df['offer_id'].isin(processed_ids)].copy()

# --- 3. ADVANCED REMEDIATION & VERIFICATION ENGINE ---
print(f"\nStep 3: Commencing advanced remediation for {len(records_to_process)} records...")
if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))
    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Processing offer_id: {row['offer_id']}")
        result_row = row.to_dict()
        # Initialize new columns
        result_row.update({'regeocoded_lat': None, 'regeocoded_lon': None, 'verified_address': None, 'remediation_method': None})

        # Process a single location (pickup or dropoff)
        def process_location(address, type_prefix):
            if pd.notna(address):
                try:
                    geocode_result = gmaps.geocode(address, bounds=cdmx_bounds)
                    if geocode_result:
                        res = geocode_result[0]
                        geom = res['geometry']

                        # --- JITTERING LOGIC ---
                        # Check if the result is an imprecise area (like a neighborhood or city)
                        if 'viewport' in geom and res['types'][0] in ['sublocality', 'locality', 'political']:
                            method = f'{type_prefix}_JITTERED_IN_BOUNDS'
                            bounds = geom['viewport']
                            sw, ne = bounds['southwest'], bounds['northeast']
                            lat = np.random.uniform(sw['lat'], ne['lat'])
                            lon = np.random.uniform(sw['lng'], ne['lng'])
                        else: # It's a precise result (rooftop, street address)
                            method = f'{type_prefix}_PRECISE'
                            loc = geom['location']
                            lat, lon = loc['lat'], loc['lng']

                        # Reverse geocode for verification
                        time.sleep(0.05)
                        reverse_res = gmaps.reverse_geocode((lat, lon))
                        verified_addr = reverse_res[0]['formatted_address'] if reverse_res else "Verification Failed"
                        return lat, lon, verified_addr, method
                except Exception:
                    return None, None, None, f'{type_prefix}_API_ERROR'
            return None, None, None, f'{type_prefix}_NO_ADDRESS'

        # Apply processing to both pickup and dropoff
        p_lat, p_lon, p_addr, p_method = process_location(row['pickup_address'], "PICKUP")
        d_lat, d_lon, d_addr, d_method = process_location(row['dropoff_address'], "DROPOFF")

        # Update the result row
        result_row.update({
            'regeocoded_pickup_lat': p_lat, 'regeocoded_pickup_lon': p_lon, 'verified_regeocoded_pickup_address': p_addr,
            'regeocoded_dropoff_lat': d_lat, 'regeocoded_dropoff_lon': d_lon, 'verified_regeocoded_dropoff_address': d_addr,
            'remediation_method': f"{p_method}, {d_method}"
        })
        results_data.append(result_row)

        if (i + 1) % 10 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Checkpoint saved.")

    print("\n--- Advanced Remediation & Verification Complete ---")
else:
    print("\nAll records have already been processed.")

# --- 4. FINAL OUTPUT ---
final_results_df = pd.DataFrame(results_data)
final_results_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nFull results saved to: {OUTPUT_FILE}")
print("Displaying a sample of the jittered verification results:")

display(final_results_df[['offer_id', 'pickup_address', 'regeocoded_pickup_lat', 'regeocoded_pickup_lon', 'verified_regeocoded_pickup_address', 'remediation_method']].head())

--- Remediation & Verification Engine v2.0 Initializing ---
Step 1: Initializing Google Maps Client and file paths...

Step 2: Loading outlier data...

Step 3: Commencing advanced remediation for 625 records...


  0%|          | 0/625 [00:00<?, ?it/s]


--- Advanced Remediation & Verification Complete ---

Full results saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/remediated_offers_verified_jittered.csv
Displaying a sample of the jittered verification results:


,offer_id,pickup_address,regeocoded_pickup_lat,regeocoded_pickup_lon,verified_regeocoded_pickup_address,remediation_method
0,OF00003,NaN,NaN,NaN,None,"PICKUP_NO_ADDRESS, DROPOFF_NO_ADDRESS"
1,OF00014,"Avenida Constituyentes, Camino Antiguo a Santa Fe",19.402747,-99.207907,"Av Constituyentes 551BIS, 16 de Septiembre, Mi...","PICKUP_PRECISE, DROPOFF_PRECISE"
2,OF00017,"Sta. Fe, Santa Fe",19.386325,-99.227191,"Vasco de Quiroga 1235, Santa Fe, Álvaro Obregó...","PICKUP_JITTERED_IN_BOUNDS, DROPOFF_PRECISE"
3,OF00019,"Av Guillermo González Camarena, Santa Fe",19.366602,-99.263853,"C. Guillermo Gonzalez Camarena 1205, Santa Fe,...","PICKUP_PRECISE, DROPOFF_PRECISE"
4,OF00020,"Calle Granada, San Fernando Huixquilucan",19.383350,-99.285252,"C. Municiones 30, San Fernando, 52765 Naucalpa...","PICKUP_JITTERED_IN_BOUNDS, DROPOFF_PRECISE"


In [ ]:
# ==============================================================================
# CELL: Full-Spectrum Geospatial Precision Audit
# ==============================================================================
# Objective: Classify every address in the dataset as high- or low-precision
# based on the result type from the Google Maps Geocoding API.

# --- 0. INSTALL DEPENDENCIES ---
!pip install googlemaps gspread google-auth-oauthlib pandas -q

import pandas as pd
import gspread
import googlemaps
import os
from google.colab import auth, userdata, drive
from tqdm.auto import tqdm
import time

print("--- Full-Spectrum Precision Audit Engine Initializing ---")

# --- 1. AUTHENTICATION & SETUP ---
drive.mount('/content/drive')
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=API_KEY)
print("Authentication successful. Google clients initialized.")

# --- 2. CONFIGURATION & FILE PATHS ---
GSHEET_NAME = 'raw_Offers'
WORKSHEET_NAME = 'diamond_offers'
ANALYSIS_PATH = '/content/drive/MyDrive/_Pienza/Assets/Phase_1'
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'geospatial_precision_audit.csv')
os.makedirs(ANALYSIS_PATH, exist_ok=True)

# --- 3. DATA INGRESS & CHECKPOINTING ---
print(f"\nStep 1: Ingesting data from '{WORKSHEET_NAME}'...")
worksheet = gc.open(GSHEET_NAME).worksheet(WORKSHEET_NAME)
all_records = worksheet.get_all_records()
df = pd.DataFrame(all_records)
df['offer_id'] = df['offer_id'].astype(str)
print(f" -> Ingested {len(df)} total records.")

processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing audit file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    print(f" -> {len(processed_ids)} records previously audited.")
records_to_process = df[~df['offer_id'].isin(processed_ids)].copy()

# --- 4. PRECISION AUDIT ENGINE ---
def get_address_precision(address):
    """
    Calls the Geocoding API and returns the precision level of the result.
    """
    if pd.isna(address) or address == '':
        return 'No_Address_Provided'
    try:
        # Using a bounds for CDMX to aid the geocoder
        geocode_result = gmaps.geocode(address, bounds={'southwest': (19.0, -99.5), 'northeast': (19.8, -98.7)})
        if geocode_result:
            location_type = geocode_result[0]['geometry'].get('location_type', 'UNKNOWN')
            if location_type in ['ROOFTOP', 'RANGE_INTERPOLATED']:
                return 'High_Precision_Point'
            else: # GEOMETRIC_CENTER or APPROXIMATE
                return 'Low_Precision_Area'
        else:
            return 'Geocode_Failed'
    except Exception:
        return 'API_Error'

print(f"\nStep 2: Commencing precision audit for {len(records_to_process)} remaining records...")
new_results = []
if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))
    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Auditing offer_id: {row['offer_id']}")

        pickup_precision = get_address_precision(row['pickup_address'])
        time.sleep(0.05) # Polite delay
        dropoff_precision = get_address_precision(row['dropoff_address'])

        new_results.append({
            'offer_id': row['offer_id'],
            'pickup_address': row['pickup_address'],
            'pickup_address_precision': pickup_precision,
            'dropoff_address': row['dropoff_address'],
            'dropoff_address_precision': dropoff_precision
        })

        # Save progress every 50 records
        if (i + 1) % 50 == 0:
            temp_df = pd.DataFrame(new_results)
            # Append to the file without headers if it's not the first write
            header = not os.path.exists(OUTPUT_FILE)
            temp_df.to_csv(OUTPUT_FILE, mode='a', header=header, index=False)
            new_results = [] # Clear the list after writing
            pbar.set_postfix_str("Checkpoint saved.")

# --- 5. FINAL EXPORT & SUMMARY ---
# Save any remaining results
if new_results:
    temp_df = pd.DataFrame(new_results)
    header = not os.path.exists(OUTPUT_FILE)
    temp_df.to_csv(OUTPUT_FILE, mode='a', header=header, index=False)

print("\n--- Audit Complete ---")
final_df = pd.read_csv(OUTPUT_FILE)
print(f"Full audit results saved to: {OUTPUT_FILE}")

print("\n--- PICKUP ADDRESS PRECISION SUMMARY ---")
print(final_df['pickup_address_precision'].value_counts())

print("\n--- DROPOFF ADDRESS PRECISION SUMMARY ---")
print(final_df['dropoff_address_precision'].value_counts())

display(final_df.head())

--- Full-Spectrum Precision Audit Engine Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Authentication successful. Google clients initialized.

Step 1: Ingesting data from 'diamond_offers'...
 -> Ingested 4766 total records.

Step 2: Commencing precision audit for 4766 remaining records...


  0%|          | 0/4766 [00:00<?, ?it/s]


--- Audit Complete ---
Full audit results saved to: /content/drive/MyDrive/_Pienza/Assets/Phase_1/geospatial_precision_audit.csv

--- PICKUP ADDRESS PRECISION SUMMARY ---
pickup_address_precision
Low_Precision_Area      4444
High_Precision_Point     310
Geocode_Failed            10
No_Address_Provided        2
Name: count, dtype: int64

--- DROPOFF ADDRESS PRECISION SUMMARY ---
dropoff_address_precision
High_Precision_Point    3566
Low_Precision_Area      1196
Geocode_Failed             3
No_Address_Provided        1
Name: count, dtype: int64


,offer_id,pickup_address,pickup_address_precision,dropoff_address,dropoff_address_precision
0,OF00001,"Calle Xochicalco, Narvarte",Low_Precision_Area,"Vía Morelos 330, Santa Clara Coatitla, 55540 E...",High_Precision_Point
1,OF00002,"Avenida Horacio, Polanco / Anzures",Low_Precision_Area,"Aut La Venta Chamapa, Ej De San Cristóbal Texc...",Low_Precision_Area
2,OF00003,NaN,No_Address_Provided,NaN,No_Address_Provided
3,OF00004,"Calle San Isidro & Privada San Isidro, Reforma...",Low_Precision_Area,"Calle Barrilaco & Calle Sierra Mojada, México ...",Low_Precision_Area
4,OF00005,"Calle Presa Don Martín, Tacuba - Azcapotzalco",Low_Precision_Area,"Calle Elpidio Cortez 300, Col Ampliación San P...",Low_Precision_Area


In [ ]:
# ==============================================================================
# CELL: Streamlined Geocoding & Verification Engine
# ==============================================================================
# Objective: Read the dedicated GSheet, perform a bounded geocode, immediately
# verify with a reverse geocode, and export the complete results.

# --- 0. INSTALL DEPENDENCIES ---
!pip install gspread google-auth-oauthlib googlemaps pandas -q

import pandas as pd
import gspread
import googlemaps
import os
from google.colab import auth, userdata, drive
from tqdm.auto import tqdm
import time

print("--- Streamlined Geocoding & Verification Engine Initializing ---")

# --- 1. SETUP & AUTHENTICATION ---
drive.mount('/content/drive')
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=API_KEY)
print("Clients initialized and Drive mounted.")

# --- 2. CONFIGURATION ---
GSHEET_NAME = 'ambiguous_addresses_updated'
ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'geocoded_results_verified.csv') # Updated filename
cdmx_bounds = {'southwest': (19.0, -99.5), 'northeast': (19.8, -98.7)}
os.makedirs(ANALYSIS_PATH, exist_ok=True)

# --- 3. DATA INGESTION & CHECKPOINTING ---
print(f"\nStep 1: Ingesting data from GSheet '{GSHEET_NAME}'...")
worksheet = gc.open(GSHEET_NAME).sheet1
all_records = worksheet.get_all_records()
source_df = pd.DataFrame(all_records)
source_df['offer_id'] = source_df['offer_id'].astype(str)
print(f" -> Ingested {len(source_df)} total records.")

results_data = []
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    results_data = pd.read_csv(OUTPUT_FILE).to_dict('records')
    print(f" -> {len(processed_ids)} records previously processed.")

records_to_process = source_df[~source_df['offer_id'].isin(processed_ids)].copy()

# --- 4. GEOCODING & VERIFICATION ENGINE ---
print(f"\nStep 2: Commencing geocode & verification for {len(records_to_process)} records...")
if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))

    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Processing offer_id: {row['offer_id']}")

        result_row = row.to_dict()
        # Add columns for both new coords and the verified address
        result_row.update({
            'geocoded_pickup_lat': None, 'geocoded_pickup_lon': None, 'verified_pickup_address': None,
            'geocoded_dropoff_lat': None, 'geocoded_dropoff_lon': None, 'verified_dropoff_address': None,
            'geocode_status': []
        })

        # --- Process Pickup Address ---
        if pd.notna(row.get('pickup_address')):
            try:
                geocode_result = gmaps.geocode(row['pickup_address'], bounds=cdmx_bounds)
                if geocode_result:
                    loc = geocode_result[0]['geometry']['location']
                    lat, lon = loc['lat'], loc['lng']
                    result_row['geocoded_pickup_lat'] = lat
                    result_row['geocoded_pickup_lon'] = lon
                    result_row['geocode_status'].append('PICKUP_SUCCESS')

                    # Immediately verify the new coordinate
                    time.sleep(0.05)
                    reverse_result = gmaps.reverse_geocode((lat, lon))
                    result_row['verified_pickup_address'] = reverse_result[0]['formatted_address'] if reverse_result else "VERIFICATION_FAILED"
                else:
                    result_row['geocode_status'].append('PICKUP_FAILED')
            except Exception:
                result_row['geocode_status'].append('PICKUP_API_ERROR')
        else:
            result_row['geocode_status'].append('PICKUP_NO_ADDRESS')

        # --- Process Dropoff Address ---
        if pd.notna(row.get('dropoff_address')):
            try:
                geocode_result = gmaps.geocode(row['dropoff_address'], bounds=cdmx_bounds)
                if geocode_result:
                    loc = geocode_result[0]['geometry']['location']
                    lat, lon = loc['lat'], loc['lng']
                    result_row['geocoded_dropoff_lat'] = lat
                    result_row['geocoded_dropoff_lon'] = lon
                    result_row['geocode_status'].append('DROPOFF_SUCCESS')

                    time.sleep(0.05)
                    reverse_result = gmaps.reverse_geocode((lat, lon))
                    result_row['verified_dropoff_address'] = reverse_result[0]['formatted_address'] if reverse_result else "VERIFICATION_FAILED"
                else:
                    result_row['geocode_status'].append('DROPOFF_FAILED')
            except Exception:
                result_row['geocode_status'].append('DROPOFF_API_ERROR')
        else:
            result_row['geocode_status'].append('DROPOFF_NO_ADDRESS')

        result_row['geocode_status'] = ', '.join(result_row['geocode_status'])
        results_data.append(result_row)

        if (i + 1) % 25 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Checkpoint saved.")
else:
    print(" -> All records have already been processed.")

# --- 5. FINAL EXPORT ---
final_df = pd.DataFrame(results_data)
final_df.to_csv(OUTPUT_FILE, index=False)
print("\n--- Geocoding & Verification Complete ---")
print(f"The results have been saved to: {OUTPUT_FILE}")
display(final_df[['offer_id', 'pickup_address', 'geocoded_pickup_lat', 'geocoded_pickup_lon', 'verified_pickup_address']].head())

--- Streamlined Geocoding & Verification Engine Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clients initialized and Drive mounted.

Step 1: Ingesting data from GSheet 'ambiguous_addresses_updated'...
 -> Ingested 625 total records.

Step 2: Commencing geocode & verification for 625 records...


  0%|          | 0/625 [00:00<?, ?it/s]


--- Geocoding & Verification Complete ---
The results have been saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/geocoded_results_verified.csv


,offer_id,pickup_address,geocoded_pickup_lat,geocoded_pickup_lon,verified_pickup_address
0,OF00003,,NaN,NaN,None
1,OF00014,"Avenida Constituyentes, Camino Antiguo a Santa...",19.402747,-99.207907,"Av Constituyentes 551BIS, 16 de Septiembre, Mi..."
2,OF00017,"Ave. Santa Fe, Santa Fe, Ciudad de Mexico, CDM...",19.362230,-99.266137,"Av Carlos Lazo 425, Santa Fe, Contadero, Cuaji..."
3,OF00019,"Av Guillermo González Camarena, Santa Fe, Ciud...",19.366602,-99.263853,"C. Guillermo Gonzalez Camarena 1205, Santa Fe,..."
4,OF00020,"Calle Granada, San Fernando Huixquilucan, Ciud...",19.360257,-99.351032,"Zaragoza 1, San Miguel, 52760 Huixquilucan de ..."


In [ ]:
# ==============================================================================
# CELL: Advanced Precision Taxonomy Audit
# ==============================================================================
# Objective: Classify every address into a sophisticated taxonomy to distinguish
# "functionally precise" areas from "truly ambiguous" ones.

# --- 0. INSTALL DEPENDENCIES ---
!pip install gspread google-auth-oauthlib googlemaps pandas -q

import pandas as pd
import gspread
import googlemaps
import os
from google.colab import auth, userdata, drive
from tqdm.auto import tqdm
import time

print("--- Advanced Precision Taxonomy Audit Initializing ---")

# --- 1. SETUP & AUTHENTICATION ---
drive.mount('/content/drive')
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=API_KEY)
print("Clients initialized and Drive mounted.")

# --- 2. CONFIGURATION ---
GSHEET_NAME = 'raw_Offers'
WORKSHEET_NAME = 'diamond_offers'
ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'precision_taxonomy_audit.csv')
cdmx_bounds = {'southwest': (19.0, -99.5), 'northeast': (19.8, -98.7)}
os.makedirs(ANALYSIS_PATH, exist_ok=True)

# --- 3. DATA INGESTION & CHECKPOINTING ---
print(f"\nStep 1: Ingesting data from GSheet '{WORKSHEET_NAME}'...")
worksheet = gc.open(GSHEET_NAME).worksheet(WORKSHEET_NAME)
source_df = pd.DataFrame(worksheet.get_all_records())
source_df['offer_id'] = source_df['offer_id'].astype(str)
print(f" -> Ingested {len(source_df)} total records.")

results_data = []
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    results_data = pd.read_csv(OUTPUT_FILE).to_dict('records')
    print(f" -> {len(processed_ids)} records previously audited.")
records_to_process = source_df[~source_df['offer_id'].isin(processed_ids)].copy()

# --- 4. ADVANCED CLASSIFICATION ENGINE ---
print(f"\nStep 2: Commencing advanced audit for {len(records_to_process)} records...")
if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))

    # Define the types that we consider "functionally precise" points
    FUNCTIONALLY_PRECISE_TYPES = {'airport', 'park', 'transit_station', 'point_of_interest', 'intersection', 'establishment'}

    def classify_address_advanced(address):
        if pd.isna(address) or address == '': return 'No_Address_Provided'
        try:
            geocode_result = gmaps.geocode(address, bounds=cdmx_bounds)
            if geocode_result:
                res = geocode_result[0]
                location_type = res['geometry'].get('location_type', 'UNKNOWN')

                # Tier 1: Highest Precision
                if location_type in ['ROOFTOP', 'RANGE_INTERPOLATED']:
                    return 'High_Precision_Point'

                # Tier 2: Functionally Precise Area (e.g., Terminal 2)
                api_types = set(res.get('types', []))
                if not FUNCTIONALLY_PRECISE_TYPES.isdisjoint(api_types):
                    return 'Functionally_Precise_Area'

                # Tier 3: Truly Ambiguous Area (e.g., Av. Javier Barros Sierra)
                return 'Ambiguous_Area'
            else:
                return 'Geocode_Failed'
        except Exception:
            return 'API_Error'

    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Auditing offer_id: {row['offer_id']}")

        pickup_precision = classify_address_advanced(row['pickup_address'])
        time.sleep(0.02)
        dropoff_precision = classify_address_advanced(row['dropoff_address'])

        results_data.append({
            'offer_id': row['offer_id'],
            'pickup_address': row['pickup_address'],
            'pickup_address_precision': pickup_precision,
            'dropoff_address': row['dropoff_address'],
            'dropoff_address_precision': dropoff_precision
        })

        if (i + 1) % 50 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Checkpoint saved.")

# --- 5. FINAL EXPORT & SUMMARY ---
final_df = pd.DataFrame(results_data)
final_df.to_csv(OUTPUT_FILE, index=False)
print("\n--- Advanced Audit Complete ---")
print(f"Full audit results saved to: {OUTPUT_FILE}")
print("\n--- PICKUP ADDRESS PRECISION SUMMARY ---")
print(final_df['pickup_address_precision'].value_counts())
print("\n--- DROPOFF ADDRESS PRECISION SUMMARY ---")
print(final_df['dropoff_address_precision'].value_counts())
display(final_df.head())

--- Advanced Precision Taxonomy Audit Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clients initialized and Drive mounted.

Step 1: Ingesting data from GSheet 'diamond_offers'...
 -> Ingested 4766 total records.

Step 2: Commencing advanced audit for 4766 records...


  0%|          | 0/4766 [00:00<?, ?it/s]


--- Advanced Audit Complete ---
Full audit results saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/precision_taxonomy_audit.csv

--- PICKUP ADDRESS PRECISION SUMMARY ---
pickup_address_precision
Ambiguous_Area               4165
High_Precision_Point          321
Functionally_Precise_Area     272
Geocode_Failed                  6
No_Address_Provided             2
Name: count, dtype: int64

--- DROPOFF ADDRESS PRECISION SUMMARY ---
dropoff_address_precision
High_Precision_Point         3568
Ambiguous_Area               1009
Functionally_Precise_Area     185
Geocode_Failed                  3
No_Address_Provided             1
Name: count, dtype: int64


,offer_id,pickup_address,pickup_address_precision,dropoff_address,dropoff_address_precision
0,OF00001,"Calle Xochicalco, Narvarte",Ambiguous_Area,"Vía Morelos 330, Santa Clara Coatitla, 55540 E...",High_Precision_Point
1,OF00002,"Avenida Horacio, Polanco / Anzures",Ambiguous_Area,"Aut La Venta Chamapa, Ej De San Cristóbal Texc...",Ambiguous_Area
2,OF00003,,No_Address_Provided,,No_Address_Provided
3,OF00004,"Calle San Isidro & Privada San Isidro, Reforma...",Functionally_Precise_Area,"Calle Barrilaco & Calle Sierra Mojada, México ...",Functionally_Precise_Area
4,OF00005,"Calle Presa Don Martín, Tacuba - Azcapotzalco",Ambiguous_Area,"Calle Elpidio Cortez 300, Col Ampliación San P...",Ambiguous_Area


In [ ]:
# ==============================================================================
# CELL: Full-Spectrum Reverse Geocode Verification
# ==============================================================================
# Objective: Create a ground-truth artifact by reverse geocoding every
# coordinate in the canonical diamond_offers sheet.

# --- 0. INSTALL DEPENDENCIES ---
!pip install gspread google-auth-oauthlib googlemaps pandas -q

import pandas as pd
import gspread
import googlemaps
import os
from google.colab import auth, userdata, drive
from tqdm.auto import tqdm
import time

print("--- Full-Spectrum Reverse Geocode Verification Initializing ---")

# --- 1. SETUP & AUTHENTICATION ---
drive.mount('/content/drive')
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=API_KEY)
print("Clients initialized and Drive mounted.")

# --- 2. CONFIGURATION ---
GSHEET_NAME = 'raw_Offers'
WORKSHEET_NAME = 'diamond_offers'
ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'diamond_offers_full_reverse_geocode_verification.csv')
os.makedirs(ANALYSIS_PATH, exist_ok=True)

# --- 3. DATA INGESTION & CHECKPOINTING ---
print(f"\nStep 1: Ingesting data from GSheet '{WORKSHEET_NAME}'...")
worksheet = gc.open(GSHEET_NAME).worksheet(WORKSHEET_NAME)
source_df = pd.DataFrame(worksheet.get_all_records())
source_df['offer_id'] = source_df['offer_id'].astype(str)
# Coerce coordinates to numeric, turning blanks/errors into NaN
for col in ['pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon']:
    source_df[col] = pd.to_numeric(source_df[col], errors='coerce')
print(f" -> Ingested {len(source_df)} total records.")

results_data = []
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    results_data = pd.read_csv(OUTPUT_FILE).to_dict('records')
    print(f" -> {len(processed_ids)} records previously processed.")

records_to_process = source_df[~source_df['offer_id'].isin(processed_ids)].copy()

# --- 4. REVERSE GEOCODING ENGINE ---
print(f"\nStep 2: Commencing full reverse geocode for {len(records_to_process)} records...")
if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))

    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Verifying offer_id: {row['offer_id']}")

        result_row = row.to_dict()

        # Process Pickup Coordinates
        if pd.notna(row['pickup_lat']) and pd.notna(row['pickup_lon']):
            try:
                reverse_result = gmaps.reverse_geocode((row['pickup_lat'], row['pickup_lon']))
                result_row['verified_pickup_address'] = reverse_result[0]['formatted_address'] if reverse_result else "No_Address_Found"
            except Exception:
                result_row['verified_pickup_address'] = 'API_Error'
        else:
            result_row['verified_pickup_address'] = 'Invalid_Or_Missing_Coords'

        time.sleep(0.02) # Polite delay

        # Process Dropoff Coordinates
        if pd.notna(row['dropoff_lat']) and pd.notna(row['dropoff_lon']):
            try:
                reverse_result = gmaps.reverse_geocode((row['dropoff_lat'], row['dropoff_lon']))
                result_row['verified_dropoff_address'] = reverse_result[0]['formatted_address'] if reverse_result else "No_Address_Found"
            except Exception:
                result_row['verified_dropoff_address'] = 'API_Error'
        else:
            result_row['verified_dropoff_address'] = 'Invalid_Or_Missing_Coords'

        results_data.append(result_row)

        if (i + 1) % 50 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Checkpoint saved.")

# --- 5. FINAL EXPORT ---
final_df = pd.DataFrame(results_data)
final_df.to_csv(OUTPUT_FILE, index=False)
print("\n--- Ground-Truth Verification Complete ---")
print(f"The definitive verification artifact has been saved to: {OUTPUT_FILE}")
display(final_df[['offer_id', 'pickup_address', 'pickup_lat', 'pickup_lon', 'verified_pickup_address']].head())

--- Full-Spectrum Reverse Geocode Verification Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clients initialized and Drive mounted.

Step 1: Ingesting data from GSheet 'diamond_offers'...
 -> Ingested 4766 total records.

Step 2: Commencing full reverse geocode for 4766 records...


  0%|          | 0/4766 [00:00<?, ?it/s]


--- Ground-Truth Verification Complete ---
The definitive verification artifact has been saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/diamond_offers_full_reverse_geocode_verification.csv


,offer_id,pickup_address,pickup_lat,pickup_lon,verified_pickup_address
0,OF00001,"Calle Xochicalco, Narvarte",19.388576,-99.154415,"Xochicalco 349, Narvarte Poniente, Benito Juár..."
1,OF00002,"Avenida Horacio, Polanco / Anzures",19.432898,-99.181742,"Calz. Gral. Mariano Escobedo 1215, Chapultepec..."
2,OF00003,,NaN,NaN,Invalid_Or_Missing_Coords
3,OF00004,"Calle San Isidro & Privada San Isidro, Reforma...",19.433370,-99.212477,"San Isidro 33, Reforma Soc, Miguel Hidalgo, 11..."
4,OF00005,"Calle Presa Don Martín, Tacuba - Azcapotzalco",19.441596,-99.209939,"C. Presa Don Martín 78, Col. Irrigación, Migue..."


In [ ]:
# ==============================================================================
# CELL: The Final Raw Metadata Extraction Engine (Ordered by offer_id)
# ==============================================================================
# Objective: For every row in the canonical dataset, extract the raw 'types'
# array from the API, preserving the original offer_id order.

# --- 0. INSTALL DEPENDENCIES ---
!pip install gspread google-auth-oauthlib googlemaps pandas -q

import pandas as pd
import gspread
import googlemaps
import os
from google.colab import auth, userdata, drive
from tqdm.auto import tqdm
import time

print("--- Final Raw Metadata Extraction Engine Initializing ---")

# --- 1. SETUP & AUTHENTICATION ---
drive.mount('/content/drive')
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=API_KEY)
print("Clients initialized and Drive mounted.")

# --- 2. CONFIGURATION ---
GSHEET_NAME = 'raw_Offers'
WORKSHEET_NAME = 'diamond_offers'
ANALYSIS_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILE = os.path.join(ANALYSIS_PATH, 'raw_api_types_audit_ordered.csv') # New, clear filename
cdmx_bounds = {'southwest': (19.0, -99.5), 'northeast': (19.8, -98.7)}
os.makedirs(ANALYSIS_PATH, exist_ok=True)

# --- 3. DATA INGESTION & CHECKPOINTING ---
print(f"\nStep 1: Ingesting data from GSheet '{WORKSHEET_NAME}'...")
worksheet = gc.open(GSHEET_NAME).worksheet(WORKSHEET_NAME)
source_df = pd.DataFrame(worksheet.get_all_records())
source_df['offer_id'] = source_df['offer_id'].astype(str)
print(f" -> Ingested {len(source_df)} total records.")

results_data = []
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(" -> Found existing results file. Resuming process...")
    results_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(results_df['offer_id'].astype(str))
    results_data = pd.read_csv(OUTPUT_FILE).to_dict('records')
    print(f" -> {len(processed_ids)} records previously analyzed.")

records_to_process = source_df[~source_df['offer_id'].isin(processed_ids)].copy()

# --- 4. METADATA EXTRACTION ENGINE (ROW-BY-ROW) ---
print(f"\nStep 2: Commencing raw metadata extraction for {len(records_to_process)} remaining rows...")
if not records_to_process.empty:
    pbar = tqdm(records_to_process.iterrows(), total=len(records_to_process))

    def get_api_types(address):
        if pd.isna(address) or address == '': return 'NO_ADDRESS'
        try:
            geocode_result = gmaps.geocode(address, bounds=cdmx_bounds)
            if geocode_result:
                api_types = geocode_result[0].get('types', [])
                return ', '.join(api_types)
            else:
                return 'GEOCODE_FAILED'
        except Exception:
            return 'API_ERROR'

    for i, (index, row) in enumerate(pbar):
        pbar.set_description(f"Processing offer_id: {row['offer_id']}")

        pickup_types = get_api_types(row['pickup_address'])
        time.sleep(0.02) # Polite delay between API calls
        dropoff_types = get_api_types(row['dropoff_address'])

        results_data.append({
            'offer_id': row['offer_id'],
            'pickup_address': row['pickup_address'],
            'pickup_api_types': pickup_types,
            'dropoff_address': row['dropoff_address'],
            'dropoff_api_types': dropoff_types
        })

        # Checkpoint every 50 records
        if (i + 1) % 50 == 0 or (i + 1) == len(records_to_process):
            pd.DataFrame(results_data).to_csv(OUTPUT_FILE, index=False)
            pbar.set_postfix_str("Checkpoint saved.")

# --- 5. FINAL EXPORT & DISPLAY ---
final_df = pd.DataFrame(results_data)
# Ensure final dataframe is sorted by the original index to guarantee order
final_df = final_df.set_index('offer_id').loc[source_df['offer_id']].reset_index()
final_df.to_csv(OUTPUT_FILE, index=False)

print("\n--- Raw Metadata Extraction Complete ---")
print(f"The ordered audit file has been saved to: {OUTPUT_FILE}")
print("\nSample of Extracted API 'types' (in original offer_id order):")
display(final_df.head(10))

--- Final Raw Metadata Extraction Engine Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clients initialized and Drive mounted.

Step 1: Ingesting data from GSheet 'diamond_offers'...
 -> Ingested 4766 total records.

Step 2: Commencing raw metadata extraction for 4766 remaining rows...


  0%|          | 0/4766 [00:00<?, ?it/s]


--- Raw Metadata Extraction Complete ---
The ordered audit file has been saved to: /content/drive/My Drive/_Pienza/Assets/Database/03_analysis/raw_api_types_audit_ordered.csv

Sample of Extracted API 'types' (in original offer_id order):


,offer_id,pickup_address,pickup_api_types,dropoff_address,dropoff_api_types
0,OF00001,"Calle Xochicalco, Narvarte",route,"Vía Morelos 330, Santa Clara Coatitla, 55540 E...","premise, street_address"
1,OF00002,"Avenida Horacio, Polanco / Anzures",route,"Aut La Venta Chamapa, Ej De San Cristóbal Texc...",route
2,OF00003,,NO_ADDRESS,,NO_ADDRESS
3,OF00004,"Calle San Isidro & Privada San Isidro, Reforma...",intersection,"Calle Barrilaco & Calle Sierra Mojada, México ...",intersection
4,OF00005,"Calle Presa Don Martín, Tacuba - Azcapotzalco",route,"Calle Elpidio Cortez 300, Col Ampliación San P...","premise, street_address"
5,OF00006,"Calle Juan Racine, Polanco / Anzures",route,"Paseo De las Aves 1, Pbo San Mateo Nopala, 532...","premise, street_address"
6,OF00007,"Calle Lago Hielmar, Tacuba - Azcapotzalco",route,"Calle Montes Urales 460, Lomas San Isidro, Ref...","premise, street_address"
7,OF00008,"Cerrada Lago Erne, Tacuba - Azcapotzalco",route,"Av Industria Militar 1055, Miguel Hidalgo 1055...",street_address
8,OF00009,"Callejón Jacarandas, Campo Militar - Naucalpan",route,"Blv Manuel Ávila Camacho 1007, Hab Jardines de...","street_address, subpremise"
9,OF00010,"Blv Manuel Avila Camacho, Polanco / Anzures",route,"Presidente Masaryk 390A, Col Polanco V Sección...",street_address
